# Session 2 · Load, inspect and clean a night of PSG

You will
1. compute **lights-off** from the subject table and see how it changes with age
2. **load** one night of polysomnography (EDF) and its expert **hypnogram** with MNE, and crop it to the night
3. **filter** the signals and compare before/after
4. **browse** the night interactively and find typical sleep patterns by eye
5. **clean** it by hand in 30-s epochs (mark bad epochs), count what you removed per stage, and compare a whole-night **spectrogram** before and after cleaning.

**How to work**
- Same as in Session 1: `Shift+Enter` runs a cell, run from top to bottom, kernel `.venv`. Replace the `...` in `# TODO` cells.
- Sections 5 and 6 open an **interactive window** outside VS Code. If you don't see it, it may be hidden behind VS Code.
- AI help (chat, inline completion) is allowed. Try first yourself, and make sure you understand every line you keep.

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.signal import welch

mne.set_log_level("WARNING")  # hide MNE's many info messages

DATA_DIR = Path("..") / "data"
RESULTS_DIR = Path("..") / "results"  # your own output files (not tracked by git), one folder per night
RESULTS_DIR.mkdir(exist_ok=True)

## 1. Lights-off

In Session 1 you saw the `LightsOff` column of `SC-subjects.csv`. It's a clock time like `"21:57:00"`. To average or plot it, we need a **number**.

The catch is **midnight**: 23:30 and 00:30 are only 1 h apart, but as hours they are 23.5 and 0.5. The trick: count hours from the midnight *before* the night, so times after midnight get +24 (00:38 → 24.63).

In [ ]:
df = pd.read_csv(DATA_DIR / "SC-subjects.csv")
lo = pd.to_datetime(df["LightsOff"], format="%H:%M:%S")  # pandas adds a dummy date (1900-01-01)
lo.head()

### Exercise 1.1: clock time → hours

In [ ]:
hours = lo.dt.hour + lo.dt.minute / 60

df["lights_off_h"] = hours.where(hours >= 12, hours + 24)

df[["subject", "night", "age", "LightsOff", "lights_off_h"]].head()

### Exercise 1.2: do older people go to bed earlier?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df["age"], df["lights_off_h"], s=15)
ticks = np.arange(21, 27)
ax.set_yticks(ticks, [f"{t % 24:02d}:00" for t in ticks])  # show hours as clock times
ax.set(xlabel="Age (years)", ylabel="Lights-off", title="Lights-off vs. age (one dot = one night)")
plt.show()

df["age_group"] = pd.cut(df["age"], bins=[0, 44, 64, 79, 120], labels=["25-44", "45-64", "65-79", "80+"])
df.groupby("age_group", observed=True)["lights_off_h"].agg(["count", "mean", "median"]).round(2)

Lights-off is a property of the **night**, so here one row per night is right (unlike age in Session 1).

⚠️ **Nights ≠ subjects.** The two nights of one person are more alike than two nights of different people. They are not independent data points. On Wednesday, when we split the data into training and test sets, both nights of a person must stay on the **same side**.

The night we use today is **subject 0, night 2**:

In [ ]:
row = df[(df["subject"] == 0) & (df["night"] == 2)]
lights_off = row["LightsOff"].iloc[0]
print("Age:", row["age"].iloc[0], "| lights-off:", lights_off)

## 2. Loading a night with MNE

Sleep-EDF file names encode the night: `SC4` + subject (2 digits) + night (1 digit). `SC4002` = subject **00**, night **2**.
Each night has two files: `SC4002E0-PSG.edf` (the signals) and `SC4002EC-Hypnogram.edf` (the expert scoring). The last two letters differ between nights, so we search with a pattern (`*`).

In [ ]:
NIGHT = "SC4002"
psg_file = next(DATA_DIR.rglob(f"{NIGHT}*-PSG.edf"))
hyp_file = next(DATA_DIR.rglob(f"{NIGHT}*-Hypnogram.edf"))

NIGHT_DIR = RESULTS_DIR / NIGHT  # your files for this night: results/SC4002/
NIGHT_DIR.mkdir(exist_ok=True)

raw = mne.io.read_raw_edf(psg_file, preload=True, infer_types=True)
raw

- `preload=True` reads the data into memory (needed to filter it).
- `infer_types=True` lets MNE guess each channel's type (EEG, EOG, …) from its name.
- The pink warnings about *different highpass/lowpass filters* are expected: the channels were recorded with different hardware filters, and MNE can store only one value.
- `raw.info` holds the metadata: channel names, sampling rate, recording date, filters, …

In [ ]:
raw.info

### Exercise 2.1: the basics of the recording

In [ ]:
sfreq = raw.info["sfreq"]

duration_h = raw.times[-1] / 3600

start = raw.info["meas_date"]

print(f"{sfreq} Hz, {duration_h:.1f} h, start {start}")
print("Highest frequency we can see (Nyquist):", sfreq / 2, "Hz")

Two things to note:
- The recording is **~24 h** long, but the night is only ~8 h of it. We crop it in Section 3.
- At 100 Hz, the highest frequency is **50 Hz**, exactly the frequency of European line noise. That's why there's no notch filter today (more on Saturday).

### Exercise 2.2: channel types

MNE uses the channel type to decide how to scale, filter and analyse each channel. `infer_types` only guesses. Check the guesses:

In [ ]:
dict(zip(raw.ch_names, raw.get_channel_types()))

| Channel | What it is |
|---|---|
| `Fpz-Cz`, `Pz-Oz` | EEG, bipolar derivations (frontal and parietal-occipital) |
| `horizontal` | EOG, horizontal eye movements |
| `submental` | chin EMG. ⚠️ In Sleep-EDF this is not the raw EMG but its **1-Hz envelope** (the muscle tone), so we never filter it like an EEG |
| `oro-nasal` | respiration (airflow) |
| `rectal` | body temperature |
| `Event marker` | a marker channel, **not** a signal |

In [ ]:
raw.set_channel_types({"Event marker": "misc"})

dict(zip(raw.ch_names, raw.get_channel_types()))

### Exercise 2.3: units

MNE stores all signals in **volts** (SI units). EEG is usually shown in **µV** (1 µV = 10⁻⁶ V). `raw.get_data()` returns a numpy array `(n_channels, n_samples)`, as in Session 1.

In [ ]:
first_30s = raw.get_data(picks="Fpz-Cz", units="uV")[:, : int(30 * sfreq)]

print(first_30s.shape)
print("Max. absolute amplitude:", np.abs(first_30s).max().round(1), "µV")

## 3. The hypnogram

The hypnogram file contains **annotations**. Each one has an `onset` (s from the recording start), a `duration` (s) and a `description` (the stage). One annotation covers a run of consecutive 30-s epochs with the same stage.

In [ ]:
annot = mne.read_annotations(hyp_file)
raw.set_annotations(annot, emit_warning=False)
raw.annotations.to_data_frame().head(8)

Sleep-EDF was scored with the old **Rechtschaffen & Kales** (R&K) rules. Today's **AASM** rules merge R&K stages 3 and 4 into **N3**. We rename the labels to the short names that **YASA** uses (the sleep-analysis package we use from Tuesday): `WAKE`, `N1`, `N2`, `N3`, `REM`, plus `ART` for artefacts (here: movement time) and `UNS` for unscored epochs.

In [ ]:
STAGE_NAMES = {
    "Sleep stage W": "WAKE",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",  # R&K stages 3 and 4 -> AASM N3
    "Sleep stage 4": "N3",
    "Sleep stage R": "REM",
    "Sleep stage ?": "UNS",  # unscored (YASA convention)
    "Movement time": "ART",  # artefact (YASA convention)
}
present = set(raw.annotations.description)
raw.annotations.rename({old: new for old, new in STAGE_NAMES.items() if old in present})
sorted(set(raw.annotations.description))

### Exercise 3.1: minutes per stage

In [ ]:
stage_minutes = pd.Series(raw.annotations.duration, index=raw.annotations.description).groupby(level=0).sum() / 60
stage_minutes.round(1)

Most of the "WAKE" is the day before and the morning after the night. We **crop** the recording:
- **start** at lights-off (from `SC-subjects.csv`)
- **end** at the end of the last sleep epoch. Sleep-EDF gives no lights-on time, so this is our choice. It changes time in bed and sleep efficiency (Saturday).

This helper converts a clock time into seconds from the start of `raw`:

In [ ]:
def clock_to_seconds(clock, raw):
    """Seconds from the first sample of `raw` to a clock time like "21:57:00"."""
    first = pd.Timestamp(raw.info["meas_date"]) + pd.Timedelta(seconds=raw.first_time)
    t = pd.Timestamp(f"{first.date()} {clock}", tz=first.tz)
    if t < first:  # e.g. 00:38 is on the next day
        t += pd.Timedelta(days=1)
    return (t - first).total_seconds()

### Exercise 3.2: crop to the night

In [ ]:
tmin = clock_to_seconds(lights_off, raw)

is_sleep = np.isin(raw.annotations.description, ["N1", "N2", "N3", "REM"])
sleep_annot = raw.annotations[is_sleep]
tmax = sleep_annot.onset[-1] + sleep_annot.duration[-1]

raw.crop(tmin=tmin, tmax=tmax)  # works in place! Re-running needs a fresh raw (reload from Section 2)
print(f"Cropped: {tmin / 3600:.2f} h to {tmax / 3600:.2f} h after the start, {raw.times[-1] / 3600:.2f} h of night")

⚠️ **Two clocks after cropping.** `raw.times` now starts at 0 = lights-off. But the annotation **onsets still count from the recording start** (`raw.first_time` is the offset between the two). Keep this in mind when you create annotations in code.

In [ ]:
print("raw.times[0] =", raw.times[0], "| raw.first_time =", raw.first_time)
raw.annotations.to_data_frame().head(3)

## 4. Filtering

We band-pass filter the EEG at **0.1–40 Hz**. The high-pass removes slow drifts (sweat, movement), the low-pass removes muscle activity and high-frequency noise. MNE uses a zero-phase FIR filter by default, so the waveforms are not shifted in time.

We filter a **copy** (as in Session 1: views vs. copies), and only the EEG and EOG channels.

The recorder already filtered the signals in hardware. The EDF header says:

| Channels | Hardware filter |
|---|---|
| `Fpz-Cz`, `Pz-Oz`, `horizontal` | high-pass 0.5 Hz, low-pass 100 Hz |
| `submental` | high-pass 16 Hz → rectified → low-pass 0.7 Hz (= the EMG envelope) |
| `oro-nasal` | 0.03–0.9 Hz |

So the EMG, respiration and temperature are left as they are.

In [ ]:
raw_filt = raw.copy().filter(l_freq=0.1, h_freq=40, picks=["eeg", "eog"])

### Exercise 4.1: before vs. after

The helper `clock_to_seconds` works on the cropped data too. Plot 30 s of `Fpz-Cz` before and after filtering.

In [ ]:
t0 = clock_to_seconds("01:00:30", raw)  # a W epoch with eye movements and muscle activity

i0 = int(t0 * sfreq)
i1 = i0 + int(30 * sfreq)

before = raw.get_data(picks="Fpz-Cz", units="uV")[0, i0:i1]
after = raw_filt.get_data(picks="Fpz-Cz", units="uV")[0, i0:i1]
t = raw.times[i0:i1] - raw.times[i0]

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True, sharey=True)
axes[0].plot(t, before, lw=0.8)
axes[0].set(ylabel="Fpz-Cz (µV)", title="01:00:30, WAKE: raw")
axes[1].plot(t, after, lw=0.8)
axes[1].set(xlabel="Time (s)", ylabel="Fpz-Cz (µV)", title="Filtered 0.1-40 Hz")
plt.show()

**Questions**
1. What did the filter change? Why so little (look at the hardware filters above)?
2. What did it *not* remove, and why not?
3. Try a narrow filter on a copy, `raw.copy().filter(1, 15, picks=["eeg", "eog"])`, and plot it instead of `after`. What disappears now?

*Answers*
1. Only the fastest part of the muscle "fuzz" (40-50 Hz). The hardware high-pass at 0.5 Hz has already removed the slow drifts, so our 0.1 Hz high-pass does nothing.
2. The large eye movements and most of the muscle activity: they lie inside 0.1-40 Hz. A filter only removes what is **outside** its band. Artefacts inside the band of interest must be rejected or repaired (Section 6, and Saturday).
3. The muscle activity is mostly gone and the signal looks "smoother", but the eye movements are still there, and so would be the beta activity and the higher part of a spindle. A narrow filter makes data look clean, not *be* clean.

### Exercise 4.2: resampling

Lowering the sampling rate makes files smaller and computations faster. Before resampling, MNE low-passes the data automatically (anti-aliasing), so frequencies above the new Nyquist frequency are removed, not folded back.

In [ ]:
raw_50 = raw_filt.copy().resample(sfreq=50)

print(len(raw_filt.times), "samples at 100 Hz ->", len(raw_50.times), "samples at 50 Hz")
# Nyquist = 25 Hz: spindles (10-16 Hz) are still there, everything above 25 Hz is gone.

We keep 100 Hz for the rest of the course.

## 5. Browsing the night

`%matplotlib qt` switches plots to interactive windows. `raw.plot()` opens the MNE browser. The hypnogram stages are shown as coloured backgrounds.

| Key / action | Does |
|---|---|
| ← / → | scroll in time |
| ↑ / ↓ | scroll through channels |
| `+` / `-` | bigger / smaller signals |
| `Home` / `End` | shorter / longer time window |
| click on the bar at the bottom | jump to that time |
| `a` | annotation mode (Exercise 5.2 and Section 6) |
| `?` | all shortcuts |

In [ ]:
CHANNELS = ["Fpz-Cz", "Pz-Oz", "horizontal", "submental"]  # the channels we look at
SCALINGS = dict(eeg=75e-6, eog=150e-6, emg=5e-6)  # signal size per channel type (V)

In [ ]:
%matplotlib qt
fig = raw_filt.plot(picks=CHANNELS, duration=30, scalings=SCALINGS, time_format="clock", title=NIGHT)

### Exercise 5.1: one example of each stage

Jump to these epochs (or open the browser at a time with `start=clock_to_seconds("01:01:00", raw_filt)`). For each one, note what you see.

| Stage | Clock time | EEG (Fpz-Cz, Pz-Oz) | EOG | EMG |
|---|---|---|---|---|
| WAKE | 01:01:00 | **alpha (~10 Hz) on Pz-Oz** (eyes closed), mixed fast activity | blinks / large eye movements | higher tone |
| N1 | 00:08:30 | low amplitude, mixed frequency (4-7 Hz), alpha gone | **slow rolling eye movements** | lower than WAKE |
| N2 | 02:56:00 | **spindles** (e.g. at ~4 s and ~15 s) and K-complexes on a theta background | quiet | low |
| N3 | 22:53:30 | **large slow waves** (0.5-2 Hz, > 75 µV), mostly on Fpz-Cz | slow waves leak into the EOG | low |
| REM | 03:39:00 | low amplitude, mixed frequency (like N1), sometimes sawtooth waves | **rapid eye movements** (first 10 s) | lowest (atonia) |

### Exercise 5.2: find them yourself

Find, by eye, somewhere **else** in the night: a **sleep spindle**, a **K-complex** and a **burst of rapid eye movements** in REM. Mark each one as an annotation:
1. Press `a` → annotation mode. In the panel, click *Add description* and create `spindle`, `K-complex` and `REMs`.
2. Select a description, then **click and drag** over the event in the signal. Drag the edges to adjust, right-click to delete.
3. Press `a` again to leave annotation mode, and close the window.

Tip: a 10-s window (`End`/`Home`) makes spindles easier to see. N2 for spindles and K-complexes (the K-complex example at 06:15:00 is ~21 s into the epoch).

In [ ]:
fig = raw_filt.plot(picks=CHANNELS, duration=30, scalings=SCALINGS, time_format="clock", title=NIGHT,
                    start=clock_to_seconds("02:56:00", raw_filt))

The annotations you made are now part of `raw_filt.annotations`. We keep yours and save them in the folder of this night:

In [ ]:
%matplotlib inline
mine = raw_filt.annotations[np.isin(raw_filt.annotations.description, ["spindle", "K-complex", "REMs"])]
mine.save(NIGHT_DIR / "events_manual.csv", overwrite=True)  # reload with mne.read_annotations(...)
mine.to_data_frame()

## 6. Cleaning by hand, epoch by epoch

Sleep-EDF is **not cleaned**: these are real home recordings with movement, muscle and electrode artefacts. Today we mark bad data by eye. On Saturday we detect it automatically and compare with your marks.

Sleep is scored in **30-s epochs**, so we clean in 30-s epochs too: an epoch is either kept or rejected as a whole. First we chop the recording into epochs.

### 6.1 Chopping the night into 30-s epochs

`mne.events_from_annotations` turns the hypnogram into **events**, one every 30 s (`chunk_duration=30`). Each event is a row `[sample, 0, stage id]`. `mne.Epochs` then cuts 30 s of signal from each event.

In [ ]:
EVENT_ID = {"WAKE": 1, "N1": 2, "N2": 3, "N3": 4, "REM": 5, "ART": 6, "UNS": 7}  # stage -> event id
events, event_id = mne.events_from_annotations(raw_filt, event_id=EVENT_ID, chunk_duration=30.0)
print(events[:3])  # sample number (from the recording start), 0, stage id

### Exercise 6.1

In [ ]:
epochs = mne.Epochs(raw_filt, events, event_id, tmin=0, tmax=30 - 1 / sfreq,
                    baseline=None, picks=CHANNELS, preload=True)
# tmax = 30 - 1/sfreq: tmin and tmax are both included, so this gives exactly 30 * sfreq = 3000 samples

print(len(epochs), "epochs of", epochs.get_data().shape[-1], "samples")
print(epochs.get_data().shape)  # (n_epochs, n_channels, n_samples)

id_to_stage = {v: k for k, v in event_id.items()}
pd.Series(epochs.events[:, 2]).map(id_to_stage).value_counts()

### 6.2 Marking bad epochs

We browse a **copy**, `epochs_clean`. In the epochs browser:
- **click on an epoch** to mark it bad (it turns red); click again to undo
- ← / → scroll, the stage of each epoch is shown at its start
- when you **close the window**, the marked epochs are dropped from `epochs_clean`. `epochs` keeps all of them.

Browse **at least the first 3 hours** and mark every epoch you would remove. Good places to start: the `ART` epoch at 22:38, and 03:11, 04:19, 05:41.

Mark only clear artefacts. A K-complex or a burst of eye movements is physiology, not an artefact! And since a whole epoch goes, ask yourself: is the artefact bad enough to lose 30 s of sleep?

In [ ]:
epochs_clean = epochs.copy()  # re-run this cell to start the marking again from scratch

In [ ]:
%matplotlib qt
fig = epochs_clean.plot(picks=CHANNELS, n_epochs=4, scalings=SCALINGS, events=True, title=f"{NIGHT}: click the bad epochs")

### Exercise 6.3: a table of the epochs

`epochs.selection` holds the original number of each epoch. Epochs that are in `epochs` but not in `epochs_clean` are the ones you dropped. We collect everything in one table, one row per epoch, with the clock time of its start, so it also works after a different crop.

In [ ]:
epoch_table = pd.DataFrame({
    "onset": pd.Timestamp(raw_filt.info["meas_date"]) + pd.to_timedelta(epochs.events[:, 0] / sfreq, unit="s"),
    "stage": pd.Series(epochs.events[:, 2]).map({v: k for k, v in event_id.items()}),
})
epoch_table["bad"] = ~np.isin(epochs.selection, epochs_clean.selection)

bad_file = NIGHT_DIR / "bad_epochs_manual.csv"
epoch_table.to_csv(bad_file, index=False)
epoch_table[epoch_table["bad"]]

### Exercise 6.4: how much did you reject, per stage?

In [ ]:
loaded = pd.read_csv(bad_file)

per_stage = loaded.groupby("stage")["bad"].agg(["size", "sum"])
per_stage.columns = ["n_epochs", "n_bad"]
per_stage["pct_rejected"] = (100 * per_stage["n_bad"] / per_stage["n_epochs"]).round(1)
per_stage

**Questions:** In which stage did you reject the most (in %)? Why could that be a problem if you later compare, say, REM between young and old subjects?

## 7. Whole-night spectrogram, before and after cleaning

A **spectrogram** (a time-frequency representation, TFR) shows how the power at each frequency changes over time. Here, one column = one 30-s epoch, and the colour = power in dB. On Monday we'll look in detail at how this is computed; today we use it to *see* the night and the artefacts.

The helpers below are given. `epoch_power` computes the power of every epoch at every frequency.

In [ ]:
TFR_CMAP = plt.get_cmap("Spectral_r").copy()  # blue = low power, red = high power
TFR_CMAP.set_bad("white")  # rejected epochs (NaN)


def epoch_power(epochs, ch="Fpz-Cz"):
    """Power (dB) of each epoch at each frequency. Returns freqs and an array (n_epochs, n_freqs)."""
    data = epochs.get_data(picks=ch, units="uV")[:, 0, :]  # (n_epochs, n_samples)
    sf = epochs.info["sfreq"]
    freqs, power = welch(data, sf, nperseg=int(4 * sf))
    return freqs, 10 * np.log10(power)


def plot_hypnogram(stages, times_h, ax):
    """The expert hypnogram as a step line, one value per epoch."""
    order = {"WAKE": 0, "REM": 1, "N1": 2, "N2": 3, "N3": 4}
    y = np.array([order.get(s, np.nan) for s in stages])
    ax.step(times_h, y, where="post", color="#0b0b0b", lw=1)
    ax.set_yticks(list(order.values()), list(order.keys()))
    ax.invert_yaxis()
    ax.set(xlabel="Time after lights-off (h)", xlim=(0, times_h[-1] + 30 / 3600))


freqs, power_before = epoch_power(epochs)
times_h = (epochs.events[:, 0] / sfreq - raw_filt.first_time) / 3600  # epoch start, h after lights-off

### Exercise 7.1

In [ ]:
%matplotlib inline

power_after = power_before.copy()
power_after[epoch_table["bad"].to_numpy()] = np.nan

# The same colour range for both panels (Why does that matter?)
vmin, vmax = np.nanpercentile(power_before[:, freqs <= 30], [2, 98])

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, height_ratios=[3, 3, 1.2])
for ax, power, title in [(axes[0], power_before, "Before cleaning"), (axes[1], power_after, "After cleaning")]:
    mesh = ax.pcolormesh(times_h, freqs, power.T, cmap=TFR_CMAP, vmin=vmin, vmax=vmax, shading="nearest")
    ax.set(ylim=(0.5, 30), ylabel="Frequency (Hz)", title=f"{title}: Fpz-Cz")
fig.colorbar(mesh, ax=axes, shrink=0.7, label="Power (dB µV²/Hz)")
plot_hypnogram(epoch_table["stage"], times_h, axes[2])
plt.show()

**Questions**
1. Compare the spectrogram with the hypnogram: which frequencies are strong in N3? In REM and WAKE?
2. What do artefacts look like in the "before" panel (which frequencies, how long)? Did you catch them all?
3. Set the colour range from `power_after` instead. What changes, and why?

## Optional exercises

1. **A loading function.** Write `load_subject(subject, night)` that returns the cropped, annotated `Raw` of one night: find the files, load, fix the channel types, add and rename the hypnogram, crop from lights-off to the last sleep epoch. Test it on night 1 (`load_subject(0, 1)`): how long is that night, and how do its stage minutes compare with night 2? *(From Saturday on, everyone gets a ready-made version of this function.)*
2. **A figure of your events.** Make a static figure (matplotlib) of the spindle, K-complex and REMs you annotated in Exercise 5.2: one panel each, 10 s around the event, EEG and EOG in µV.

In [ ]:

# Optional exercise 1
def load_subject(subject, night, data_dir=DATA_DIR):
    """Load one Sleep-EDF night, cropped from lights-off to the end of the last sleep epoch."""
    code = f"SC4{subject:02d}{night}"
    raw = mne.io.read_raw_edf(next(data_dir.rglob(f"{code}*-PSG.edf")), preload=True, infer_types=True)
    raw.set_channel_types({"Event marker": "misc"})
    raw.set_annotations(mne.read_annotations(next(data_dir.rglob(f"{code}*-Hypnogram.edf"))), emit_warning=False)
    present = set(raw.annotations.description)
    raw.annotations.rename({old: new for old, new in STAGE_NAMES.items() if old in present})

    subjects = pd.read_csv(data_dir / "SC-subjects.csv")
    lights_off = subjects.loc[(subjects["subject"] == subject) & (subjects["night"] == night), "LightsOff"].iloc[0]
    sleep = raw.annotations[np.isin(raw.annotations.description, ["N1", "N2", "N3", "REM"])]
    raw.crop(tmin=clock_to_seconds(lights_off, raw), tmax=sleep.onset[-1] + sleep.duration[-1])
    return raw


raw1 = load_subject(0, 1)
minutes = pd.Series(raw1.annotations.duration, index=raw1.annotations.description).groupby(level=0).sum() / 60
print(f"Night 1: {raw1.times[-1] / 3600:.2f} h")
minutes.round(1)


In [ ]:
# Optional exercise 2
events = mine if len(mine) else mne.Annotations([43563, 55519, 46140], [1, 1, 10], ["spindle", "K-complex", "REMs"],
                                                orig_time=raw_filt.annotations.orig_time)
fig, axes = plt.subplots(len(events), 1, figsize=(10, 2.5 * len(events)))
for ax, ev in zip(np.atleast_1d(axes), events):
    center = ev["onset"] - raw_filt.first_time + ev["duration"] / 2  # the two clocks again
    i0, i1 = int((center - 5) * sfreq), int((center + 5) * sfreq)
    t = np.arange(i1 - i0) / sfreq - 5
    for ch, offset in [("Fpz-Cz", 0), ("Pz-Oz", -150), ("horizontal", -300)]:
        ax.plot(t, raw_filt.get_data(picks=ch, units="uV")[0, i0:i1] + offset, lw=0.8, label=ch)
    ax.set(title=ev["description"], ylabel="µV (offset)")
ax.set(xlabel="Time from event centre (s)")
axes[0].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()